# 使用客户端工具创建客户服务代理

在这个教程中,我们将演示如何使用 Claude 3 和客户端工具创建一个客户服务聊天机器人。该聊天机器人将能够查询客户信息、获取订单详情,并代表客户取消订单。我们将定义必要的工具,并模拟合成响应来展示聊天机器人的功能。

## 步骤 1: 设置环境

首先,让我们安装所需的库并设置 Claude API 客户端。

In [ ]:
%pip install anthropic

In [18]:
import anthropic

client = anthropic.Client()
MODEL_NAME = "claude-opus-4-1"

## 步骤 2: 定义客户端工具

接下来,我们将定义聊天机器人将用于协助客户的客户端工具。我们将创建三个工具:get_customer_info、get_order_details 和 cancel_order。

In [2]:
tools = [
    {
        "name": "get_customer_info",
        "description": "根据客户 ID 检索客户信息。返回客户的姓名、电子邮件和电话号码。",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "客户的唯一标识符。",
                }
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "get_order_details",
        "description": "根据订单 ID 检索特定订单的详细信息。返回订单 ID、产品名称、数量、价格和订单状态。",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "订单的唯一标识符。",
                }
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "cancel_order",
        "description": "根据提供的订单 ID 取消订单。如果取消成功,则返回确认消息。",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "要取消的订单的唯一标识符。",
                }
            },
            "required": ["order_id"],
        },
    },
]

## 步骤 3: 模拟合成工具响应

由于我们没有真实的客户数据或订单信息,我们将为工具模拟合成响应。在实际场景中,这些函数将与您的实际客户数据库和订单管理系统交互。

In [15]:
def get_customer_info(customer_id):
    # 模拟客户数据
    customers = {
        "C1": {"name": "John Doe", "email": "john@example.com", "phone": "123-456-7890"},
        "C2": {"name": "Jane Smith", "email": "jane@example.com", "phone": "987-654-3210"},
    }
    return customers.get(customer_id, "Customer not found")


def get_order_details(order_id):
    # 模拟订单数据
    orders = {
        "O1": {
            "id": "O1",
            "product": "Widget A",
            "quantity": 2,
            "price": 19.99,
            "status": "Shipped",
        },
        "O2": {
            "id": "O2",
            "product": "Gadget B",
            "quantity": 1,
            "price": 49.99,
            "status": "Processing",
        },
    }
    return orders.get(order_id, "Order not found")


def cancel_order(order_id):
    # 模拟订单取消
    if order_id in ["O1", "O2"]:
        return True
    else:
        return False

## 步骤 4: 处理工具调用并返回结果

我们将创建一个函数来处理 Claude 发起的工具调用,并返回适当的结果。

In [4]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "get_customer_info":
        return get_customer_info(tool_input["customer_id"])
    elif tool_name == "get_order_details":
        return get_order_details(tool_input["order_id"])
    elif tool_name == "cancel_order":
        return cancel_order(tool_input["order_id"])

## 步骤 5: 与聊天机器人交互

现在,让我们创建一个函数来与聊天机器人交互。我们将发送用户消息,处理 Claude 发起的任何工具调用,并向用户返回最终响应。

In [13]:
import json


def chatbot_interaction(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {response.stop_reason}")
    print(f"Content: {response.content}")

    while response.stop_reason == "tool_use":
        tool_use = next(block for block in response.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print("Tool Input:")
        print(json.dumps(tool_input, indent=2))

        tool_result = process_tool_call(tool_name, tool_input)

        print("\nTool Result:")
        print(json.dumps(tool_result, indent=2))

        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": response.content},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use.id,
                        "content": str(tool_result),
                    }
                ],
            },
        ]

        response = client.messages.create(
            model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
        )

        print("\nResponse:")
        print(f"Stop Reason: {response.stop_reason}")
        print(f"Content: {response.content}")

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )

    print(f"\nFinal Response: {final_response}")

    return final_response

## 步骤 6: 测试聊天机器人
让我们用几个示例查询测试我们的客户服务聊天机器人。

In [17]:
chatbot_interaction("Can you tell me the email address for customer C1?")
chatbot_interaction("What is the status of order O2?")
chatbot_interaction("Please cancel order O1 for me.")


User Message: Can you tell me the email address for customer C1?

Initial Response:
Stop Reason: tool_use
Content: [ContentBlock(text='<thinking>The get_customer_info function retrieves a customer\'s name, email, and phone number given their customer ID. To call this function, I need the customer_id parameter. The user provided the customer ID "C1" in their request, so I have the necessary information to make the API call.</thinking>', type='text'), ContentBlockToolUse(id='toolu_019F9JHokMkJ1dHw5BEh28sA', input={'customer_id': 'C1'}, name='get_customer_info', type='tool_use')]

Tool Used: get_customer_info
Tool Input:
{
  "customer_id": "C1"
}

Tool Result:
{
  "name": "John Doe",
  "email": "john@example.com",
  "phone": "123-456-7890"
}

Response:
Stop Reason: end_turn
Content: [ContentBlock(text='The email address for customer C1 (John Doe) is john@example.com.', type='text')]

Final Response: The email address for customer C1 (John Doe) is john@example.com.

User Message: What is 

'Based on the confirmation received, your order O1 has been successfully cancelled. Please let me know if there is anything else I can assist you with.'

就这样!我们使用 Claude 3 模型和客户端工具创建了一个客户服务聊天机器人。聊天机器人可以根据用户的请求查询客户信息、获取订单详情和取消订单。通过定义清晰的工具描述和输入模式,我们使 Claude 能够有效地理解并使用可用工具来协助客户。

请随意扩展此示例,将其与您的实际客户数据库和订单管理系统集成,并添加更多工具来处理更广泛的客户服务任务。